# Notebook 02 — Baseline U-Net (Image-Only) · v2
**Model A: U-Net with CBAM attention + improved training pipeline**

Key upgrades:
- CBAM channel+spatial attention on all skip connections
- DropBlock bottleneck regularisation
- AdamW + cosine annealing with warm restarts
- Linear LR warmup (5 epochs)
- Gradient accumulation (effective batch = 8)
- SymmetricUnifiedFocalLoss (best for class-imbalanced segmentation)
- Richer SAR augmentations (speckle noise, grid distortion, cutout)
- Test-time augmentation (TTA) for evaluation

In [ ]:
import sys, os
os.chdir(r'd:\flood_segmentation')
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/flood_segmentation')
    %pip install -q rasterio tqdm scikit-learn scipy

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('.'))

from src.datasets import build_datasets
from src.models   import get_model
from src.train    import train_model, evaluate_model
from src.utils    import visualize_predictions, plot_training_history

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Hyperparameters ─────────────────────────────────────────────────────────
IMG_SIZE    = 256
BATCH_SIZE  = 4      # physical batch; effective = BATCH_SIZE * ACCUM_STEPS = 8
ACCUM_STEPS = 2      # gradient accumulation (keeps GPU memory low)
NUM_EPOCHS  = 60
LR          = 3e-4   # peak lr after warmup
PATIENCE    = 15
WARMUP      = 5      # linear warmup epochs
COSINE_T0   = 20     # cosine restart period
NUM_WORKERS = 0      # 0 on Windows; 4 on Linux

DATA_DIR    = 'data/raw'
WEATHER_CSV = 'data/processed/weather.csv'
SAVE_PATH   = 'results/saved_models/baseline_unet_v2.pth'
FIG_DIR     = 'results/figures'
os.makedirs(FIG_DIR, exist_ok=True)
print('Config loaded.')

In [ ]:
# --- Build datasets ---
train_ds, val_ds, test_ds = build_datasets(
    data_dir=DATA_DIR,
    weather_csv_path=WEATHER_CSV,
    img_size=IMG_SIZE,
    train_ratio=0.70,
    val_ratio=0.15,
    seed=42,
)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

# Sanity check a single sample
img, weather, mask = train_ds[0]
print(f'Image shape  : {img.shape}   dtype: {img.dtype}')
print(f'Weather shape: {weather.shape}')
print(f'Mask shape   : {mask.shape}  flood%: {mask.mean().item()*100:.1f}%')

In [ ]:
# --- Build Model A: Baseline U-Net with CBAM attention ---
model_A = get_model('baseline', img_ch=2, base_ch=64, drop_prob=0.1)
n_params = sum(p.numel() for p in model_A.parameters() if p.requires_grad)
print(f'BaselineUNet (v2 CBAM) — Trainable parameters: {n_params:,}')

In [ ]:
# --- Train ---
history_A, best_iou_A = train_model(
    model=model_A,
    train_dataset=train_ds,
    val_dataset=val_ds,
    is_multimodal=False,
    save_path=SAVE_PATH,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    device=device,
    warmup_epochs=WARMUP,
    accum_steps=ACCUM_STEPS,
    cosine_T0=COSINE_T0,
)
print(f'\nBest Val IoU (Baseline U-Net v2): {best_iou_A:.4f}')

In [ ]:
# --- Training curves ---
fig = plot_training_history(
    history_A,
    save_path=f'{FIG_DIR}/02_baseline_v2_history.png',
    title='Baseline U-Net v2 — Training History',
)
plt.show()

In [ ]:
# --- Evaluate on test set (with TTA) ---
test_metrics_A = evaluate_model(
    model=get_model('baseline', img_ch=2, base_ch=64, drop_prob=0.1),
    test_dataset=test_ds,
    checkpoint_path=SAVE_PATH,
    is_multimodal=False,
    batch_size=BATCH_SIZE,
    device=device,
    use_tta=True,
)

# Save for ablation study
import json
os.makedirs('results', exist_ok=True)
with open('results/metrics_baseline.json', 'w') as f:
    json.dump(test_metrics_A, f, indent=2)
print('Test metrics saved to results/metrics_baseline.json')

In [ ]:
# --- Qualitative predictions ---
from torch.utils.data import DataLoader

model_A.eval()
model_A = model_A.to(device)

loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=0)
imgs, weather, masks = next(iter(loader))
with torch.no_grad():
    # Model outputs raw logits — apply sigmoid before visualisation
    preds = torch.sigmoid(model_A(imgs.to(device))).cpu()

fig = visualize_predictions(
    imgs, masks, preds, n_samples=4,
    save_path=f'{FIG_DIR}/02_baseline_v2_predictions.png',
    title='Baseline U-Net v2 Predictions',
)
plt.show()
print('Notebook 02 complete. Proceed to Notebook 03 for the multimodal model.')